<a href="https://colab.research.google.com/github/dung-ho03/capdyssey/blob/main/but_ppg_quality_cnn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BUT PPG 신호 품질 1D-CNN — 개발 · 평가

| 항목 | 내용 |
|---|---|
| 입력 | PPG (스마트폰 영상 **빨간 채널** 기반), **10초 · 30Hz → 300개 값** |
| 정답 | `quality-hr-ann.csv` — **1 = 심박수 읽기 좋은 품질 / 0 = 나쁜 품질** |
| 연결 | 정답 CSV의 **ID** = PPG 파일의 ID (`{ID}/{ID}_PPG.dat/.hea`) |
| 분할 | **피험자 단위** (ID 앞 3자리 = 피험자, 뒤 3자리 = 측정 번호) 5-fold 교차검증 |

흐름: 레포 데이터 다운로드 확인 → 로드/ID 매칭 → 전처리 → 5-fold CV(1D-CNN + 간단 baseline) → 평가

## 0. 환경 · 데이터 준비

In [ ]:
# 이미 레포를 clone/다운로드 했다면 이 셀은 건너뛰어도 됩니다.
# !git clone https://github.com/<계정>/<레포>.git && %cd <레포>
# !python -m pip install -r requirements.txt
# !python scripts/download_data.py but      # ← 실제 BUT PPG 데이터셋 이름 확인: --list
!pip -q install wfdb scikit-learn scipy matplotlib pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os, glob, json, random
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt
from scipy.stats import skew, kurtosis
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_recall_fscore_support,
                             roc_auc_score, average_precision_score, confusion_matrix, roc_curve,
                             precision_recall_curve, matthews_corrcoef)
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 1. 설정

In [ ]:
USE_SYNTHETIC = False                 # True: 데이터 없이 파이프라인만 점검
DATA_ROOT     = "/content"            # quality-hr-ann.csv 를 이 아래에서 재귀 탐색합니다
FS, N_SAMPLES = 30, 300               # 30Hz x 10초
CHANNEL_NAME  = "PPG_R"               # 헤더에 이 이름의 채널이 있으면 사용, 없으면 첫 채널
N_FOLDS       = 5
BATCH, EPOCHS, LR, PATIENCE = 64, 60, 1e-3, 10

## 2. 데이터 로드 · ID 연결
- 정답 CSV: `ID, quality, HR` (3열). 헤더 유무는 자동 처리합니다.
- PPG 경로: `{ID}/{ID}_PPG` (WFDB). 같은 ID끼리 연결하고, 파일이 없는 ID는 목록으로 보여줍니다.
- 피험자 = `ID[:3]` → 분할 기준(누수 방지).

In [ ]:
def synthetic_records(n_subj=50, per_subj=40):
    rng = np.random.default_rng(SEED); t = np.arange(N_SAMPLES)/FS; rows = []
    for s in range(n_subj):
        for m in range(per_subj):
            hr = rng.uniform(55, 110)/60
            x = np.sin(2*np.pi*hr*t + rng.uniform(0, 6.28)) + 0.4*np.sin(4*np.pi*hr*t) + 0.05*rng.standard_normal(N_SAMPLES)
            good = rng.random() > 0.35
            if not good: x += rng.normal(0, rng.uniform(0.8, 2.0), N_SAMPLES) + rng.uniform(0, 2)*np.sin(2*np.pi*rng.uniform(.1, .5)*t)
            rows.append((f"{100+s:03d}{m+1:03d}", x.astype(np.float32), int(good)))
    return pd.DataFrame(rows, columns=["ID", "ppg", "quality"])

def load_but_ppg():
    ann_files = glob.glob(os.path.join(DATA_ROOT, "**", "quality-hr-ann.csv"), recursive=True)
    assert ann_files, f"{DATA_ROOT} 아래에서 quality-hr-ann.csv 를 찾지 못했습니다. 다운로드 경로를 확인하세요."
    ann_path = ann_files[0]; base = os.path.dirname(ann_path); print("annotation:", ann_path)

    df = pd.read_csv(ann_path, header=None)
    if not str(df.iloc[0, 0]).strip().isdigit():          # 첫 행이 헤더면 제거
        df = df.iloc[1:]
    df = df.iloc[:, :3]; df.columns = ["ID", "quality", "HR"]
    df["ID"] = df["ID"].astype(str).str.strip().str.replace(r"\.0$", "", regex=True).str.zfill(6)
    df["quality"] = df["quality"].astype(int)

    ppgs, keep, missing, lens, chans = [], [], [], [], set()
    for rid in df["ID"]:
        path = os.path.join(base, rid, f"{rid}_PPG")
        if not os.path.exists(path + ".dat"):
            missing.append(rid); ppgs.append(None); continue
        rec = wfdb.rdrecord(path)
        assert round(rec.fs) == FS, f"{rid}: fs={rec.fs}"
        names = [n.upper() for n in rec.sig_name]; chans.add(tuple(rec.sig_name))
        ch = names.index(CHANNEL_NAME) if CHANNEL_NAME in names else 0
        x = rec.p_signal[:, ch].astype(np.float32)
        lens.append(len(x))
        x = x[:N_SAMPLES] if len(x) >= N_SAMPLES else np.pad(x, (0, N_SAMPLES-len(x)), mode="edge")
        ppgs.append(x)
    df["ppg"] = ppgs
    print("헤더 채널명:", chans)
    print("길이 분포(min/median/max):", (min(lens), int(np.median(lens)), max(lens)))
    print(f"CSV {len(df)}개 중 PPG 파일 없음: {len(missing)}개", missing[:10])
    return df.dropna(subset=["ppg"]).reset_index(drop=True)

data = synthetic_records() if USE_SYNTHETIC else load_but_ppg()
data["subject"] = data["ID"].str[:3]
X_raw = np.stack(data["ppg"].values); y = data["quality"].values.astype(int); groups = data["subject"].values
print(f"samples={len(data)} subjects={len(set(groups))} | 좋음(1) 비율={y.mean():.3f} | X={X_raw.shape}")

AssertionError: /content 아래에서 quality-hr-ann.csv 를 찾지 못했습니다. 다운로드 경로를 확인하세요.

## 3. 전처리 (0.5–5Hz 밴드패스 → 샘플별 z-score)

In [ ]:
b, a = butter(3, [0.5/(FS/2), 5.0/(FS/2)], btype="band")
def preprocess(x):
    x = filtfilt(b, a, x - x.mean())
    return ((x - x.mean()) / (x.std() + 1e-8)).astype(np.float32)

X = np.stack([preprocess(x) for x in X_raw])[:, None, :]     # (N, 1, 300)
fig, ax = plt.subplots(2, 3, figsize=(14, 4))
for r, lab in enumerate([1, 0]):
    for c, i in enumerate(np.where(y == lab)[0][:3]):
        ax[r, c].plot(X[i, 0]); ax[r, c].set_title(f"quality={lab}  ID={data['ID'][i]}", fontsize=9)
plt.tight_layout(); plt.show()

## 4. 모델 · 학습 함수

In [ ]:
class Block(nn.Module):
    def __init__(self, cin, cout, k=5, p=0.15):
        super().__init__()
        self.net = nn.Sequential(nn.Conv1d(cin, cout, k, padding=k//2), nn.BatchNorm1d(cout), nn.ReLU(),
                                 nn.Conv1d(cout, cout, k, padding=k//2), nn.BatchNorm1d(cout), nn.ReLU(),
                                 nn.MaxPool1d(2), nn.Dropout(p))
    def forward(self, x): return self.net(x)

class PPGQualityCNN(nn.Module):               # 입력 (B,1,300) → 좋은 품질일 로짓
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(Block(1, 16), Block(16, 32), Block(32, 64))
        self.head = nn.Sequential(nn.AdaptiveAvgPool1d(1), nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Dropout(0.3), nn.Linear(32, 1))
    def forward(self, x): return self.head(self.features(x)).squeeze(-1)

def make_loader(idx, shuffle=False):
    ds = TensorDataset(torch.tensor(X[idx]), torch.tensor(y[idx], dtype=torch.float32))
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle)

@torch.no_grad()
def predict(model, idx):
    model.eval(); out = []
    for xb, _ in make_loader(idx):
        out.append(torch.sigmoid(model(xb.to(device))).cpu().numpy())
    return np.concatenate(out)

def train_one(tr_idx, va_idx, tag=""):
    torch.manual_seed(SEED)
    model = PPGQualityCNN().to(device)
    pos_w = torch.tensor((1 - y[tr_idx].mean()) / max(y[tr_idx].mean(), 1e-6), device=device)   # 클래스 불균형 보정
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)
    dl = make_loader(tr_idx, True); best, wait, best_state, hist = -1, 0, None, []
    for ep in range(EPOCHS):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            xb = xb + 0.02*torch.randn_like(xb)             # 가벼운 노이즈 증강
            opt.zero_grad(); crit(model(xb), yb).backward(); opt.step()
        ap = average_precision_score(y[va_idx], predict(model, va_idx)); sched.step(ap); hist.append(ap)
        if ap > best: best, wait, best_state = ap, 0, {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE: break
    model.load_state_dict(best_state)
    print(f"{tag} epochs={ep+1} best val AUPRC={best:.4f}")
    return model, hist

## 5. 5-fold 피험자 단위 교차검증
- 각 fold에서 test 피험자는 학습에 전혀 쓰이지 않습니다.
- train 피험자 중 20%를 val로 떼어 early stopping과 **임계값(F1 최대)** 을 정하고, 그 임계값을 test에 그대로 적용합니다.
- 비교용 baseline: 단순 통계 특징 + 로지스틱 회귀 (같은 분할)

In [ ]:
def baseline_features(x):                    # x: 전처리된 1D 신호
    f = np.abs(np.fft.rfft(x))**2; fr = np.fft.rfftfreq(len(x), 1/FS)
    band = (fr >= 0.7) & (fr <= 3.5)
    ac = np.correlate(x, x, "full")[len(x)-1:]; ac = ac/ac[0]
    return [skew(x), kurtosis(x), f[band].max()/(f.sum()+1e-9), f[band].sum()/(f.sum()+1e-9), ac[8:45].max(), np.abs(np.diff(x)).mean()]
F = np.array([baseline_features(x[0]) for x in X])

def best_threshold(y_true, p):
    pr, rc, th = precision_recall_curve(y_true, p); f1 = 2*pr*rc/(pr+rc+1e-9)
    return float(th[np.argmax(f1[:-1])])

skf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
p_cnn = np.zeros(len(y)); p_base = np.zeros(len(y)); thr_cnn = np.zeros(len(y)); thr_base = np.zeros(len(y)); fold_id = np.zeros(len(y), int)

for k, (trval, te) in enumerate(skf.split(X, y, groups)):
    tr_rel, va_rel = next(GroupShuffleSplit(1, test_size=0.2, random_state=SEED+k).split(trval, y[trval], groups[trval]))
    tr, va = trval[tr_rel], trval[va_rel]
    assert not set(groups[trval]) & set(groups[te])
    print(f"[fold {k}] train={len(tr)} val={len(va)} test={len(te)} | test subjects={len(set(groups[te]))}")
    model, _ = train_one(tr, va, f"[fold {k}]")
    p_cnn[te] = predict(model, te); thr_cnn[te] = best_threshold(y[va], predict(model, va))
    clf = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(F[tr], y[tr])
    p_base[te] = clf.predict_proba(F[te])[:, 1]; thr_base[te] = best_threshold(y[va], clf.predict_proba(F[va])[:, 1])
    fold_id[te] = k
    torch.save(model.state_dict(), f"ppg_quality_fold{k}.pt")

## 6. 평가

In [ ]:
def metrics(y_true, p, thr):
    pred = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
    return {"AUROC": roc_auc_score(y_true, p), "AUPRC": average_precision_score(y_true, p),
            "Acc": accuracy_score(y_true, pred), "BalAcc": balanced_accuracy_score(y_true, pred),
            "Precision(good)": pr, "Recall(good)": rc, "F1(good)": f1,
            "Specificity(bad감지)": tn/(tn+fp+1e-9), "MCC": matthews_corrcoef(y_true, pred)}

summary = pd.DataFrame({"1D-CNN": metrics(y, p_cnn, thr_cnn), "Baseline(LogReg)": metrics(y, p_base, thr_base)}).round(4)
display(summary)

rows = []
for k in range(N_FOLDS):
    m = fold_id == k; r = metrics(y[m], p_cnn[m], thr_cnn[m]); r["fold"] = k; rows.append(r)
cv = pd.DataFrame(rows).set_index("fold"); display(cv.round(4)); print(cv.agg(["mean", "std"]).round(4).loc[["mean", "std"], ["AUROC", "AUPRC", "BalAcc", "F1(good)"]])

In [ ]:
pred = (p_cnn >= thr_cnn).astype(int)
fig, ax = plt.subplots(1, 4, figsize=(18, 4))
for name, p in [("1D-CNN", p_cnn), ("Baseline", p_base)]:
    fpr, tpr, _ = roc_curve(y, p); ax[0].plot(fpr, tpr, label=f"{name} ({roc_auc_score(y, p):.3f})")
    pr_, rc_, _ = precision_recall_curve(y, p); ax[1].plot(rc_, pr_, label=f"{name} ({average_precision_score(y, p):.3f})")
ax[0].plot([0,1],[0,1],"--",c="gray"); ax[0].set_title("ROC"); ax[0].legend()
ax[1].set_title("PR (positive = good)"); ax[1].set_xlabel("recall"); ax[1].set_ylabel("precision"); ax[1].legend()
cm = confusion_matrix(y, pred); ax[2].imshow(cm, cmap="Blues"); ax[2].set_title("Confusion (0=bad, 1=good)")
for i in range(2):
    for j in range(2): ax[2].text(j, i, cm[i, j], ha="center", va="center")
ax[2].set_xticks([0,1]); ax[2].set_yticks([0,1])
bins = np.linspace(0, 1, 11); bi = np.digitize(p_cnn, bins) - 1
xs = [p_cnn[bi==c].mean() for c in range(10) if (bi==c).sum() > 5]; ys_ = [y[bi==c].mean() for c in range(10) if (bi==c).sum() > 5]
ax[3].plot(xs, ys_, "o-"); ax[3].plot([0,1],[0,1],"--",c="gray"); ax[3].set_title("Calibration"); ax[3].set_xlabel("pred P(good)"); ax[3].set_ylabel("observed good ratio")
plt.tight_layout(); plt.show()

# 피험자별 성능
rows = []
for s in sorted(set(groups)):
    m = groups == s
    rows.append({"subject": s, "n": m.sum(), "good_ratio": y[m].mean(), "acc": accuracy_score(y[m], pred[m]),
                 "AUROC": roc_auc_score(y[m], p_cnn[m]) if len(set(y[m])) > 1 else np.nan})
subj = pd.DataFrame(rows); print(subj[["acc", "AUROC"]].describe().round(3)); display(subj.sort_values("acc").head(8).round(3))

# 오분류 예시
wrong = np.where(pred != y)[0]; wrong = wrong[np.argsort(-np.abs(p_cnn[wrong] - y[wrong]))][:6]   # 가장 확신하며 틀린 순
fig, axes = plt.subplots(len(wrong), 1, figsize=(10, 1.8*max(len(wrong), 1)), squeeze=False)
for a, i in zip(axes[:, 0], wrong): a.plot(X[i, 0]); a.set_title(f"ID={data['ID'][i]} true={y[i]} P(good)={p_cnn[i]:.2f}", fontsize=9)
plt.tight_layout(); plt.show()

summary.to_csv("but_ppg_cv_summary.csv"); cv.to_csv("but_ppg_cv_per_fold.csv")
pd.DataFrame({"ID": data["ID"], "subject": groups, "y": y, "p_cnn": p_cnn, "thr": thr_cnn, "fold": fold_id}).to_csv("but_ppg_oof_predictions.csv", index=False)